# Notebook 02: Exploratory Data Analysis & Feature Engineering

This notebook performs comprehensive EDA on the UMKM dataset generated in Notebook 01, followed by feature engineering to create derived variables for subsequent modeling.

**Sections:**
1. Load Data & Initial Inspection
2. Univariate Analysis
3. Spatial Distribution Analysis
4. Correlation Analysis
5. Bivariate Analysis
6. Spatial Autocorrelation
7. Feature Engineering
8. Feature Selection
9. Save Engineered Dataset

## 1. Load Data & Initial Inspection

We load the UMKM dataset (10,000 records) and kecamatan reference data generated by Notebook 01.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# Load datasets
df = pd.read_csv('../data/umkm_dataset.csv')
kecamatan_ref = pd.read_csv('../data/kecamatan_reference.csv')

print(f"UMKM Dataset Shape: {df.shape}")
print(f"Kecamatan Reference Shape: {kecamatan_ref.shape}")
print()
print("=== UMKM Dataset Info ===")
print(df.dtypes)

UMKM Dataset Shape: (10000, 24)
Kecamatan Reference Shape: (596, 17)

=== UMKM Dataset Info ===
kabupaten_kota                   object
kecamatan                        object
latitude                        float64
longitude                       float64
is_kota                            bool
jenis_usaha                      object
tahun_berdiri                     int64
jumlah_karyawan                   int64
has_digital_presence              int64
omset_bulanan                   float64
populasi                          int64
kepadatan_penduduk              float64
income_per_kapita               float64
jarak_ke_jalan_utama            float64
jarak_ke_pasar                  float64
akses_internet_pct              float64
skor_infrastruktur              float64
jumlah_kompetitor_radius_3km      int64
jarak_ke_bank_terdekat          float64
penetrasi_kur_pct               float64
risiko_banjir                   float64
risiko_gempa                    float64
skor_potensi            

In [3]:
# Missing values summary
print("=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
print(missing_df[missing_df['Count'] > 0] if missing_df['Count'].sum() > 0 else "No missing values found!")
print()
print("=== First 5 Rows ===")
df.head()

=== Missing Values ===
No missing values found!

=== First 5 Rows ===


,kabupaten_kota,kecamatan,latitude,longitude,is_kota,jenis_usaha,tahun_berdiri,jumlah_karyawan,has_digital_presence,omset_bulanan,...,jarak_ke_pasar,akses_internet_pct,skor_infrastruktur,jumlah_kompetitor_radius_3km,jarak_ke_bank_terdekat,penetrasi_kur_pct,risiko_banjir,risiko_gempa,skor_potensi,is_survived_3yr
0,Kab. Bandung,Cileunyi,-6.995987,107.668169,False,Makanan,2010,2,1,21.74,...,5.98,65.8,72.0,0,6.65,18.0,0.05,0.349,62.99,1
1,Kab. Bandung,Cileunyi,-6.997196,107.663126,False,Makanan,2010,6,1,150.48,...,5.98,65.8,72.0,0,6.65,18.0,0.05,0.349,62.99,1
2,Kab. Bandung,Cileunyi,-7.002215,107.671349,False,Makanan,2022,14,0,30.48,...,5.98,65.8,72.0,0,6.65,18.0,0.05,0.349,55.40,0
3,Kab. Bandung,Cileunyi,-6.999372,107.665906,False,Fashion,2020,4,1,75.89,...,5.98,65.8,72.0,0,6.65,18.0,0.05,0.349,62.99,0
4,Kab. Bandung,Cileunyi,-7.003616,107.661739,False,Kerajinan,2019,4,0,62.11,...,5.98,65.8,72.0,0,6.65,18.0,0.05,0.349,55.40,0


In [4]:
# Summary statistics for all numeric columns
print("=== Summary Statistics (Numeric Columns) ===")
df.describe().T

=== Summary Statistics (Numeric Columns) ===


,count,mean,std,min,25%,50%,75%,max
latitude,10000.0,-6.771159,0.341655,-7.756945,-6.970055,-6.818983,-6.514862,-6.155012
longitude,10000.0,107.624015,0.605682,106.704086,107.007954,107.602674,108.221281,108.719185
tahun_berdiri,10000.0,2017.028200,4.329004,2010.000000,2013.000000,2017.000000,2021.000000,2024.000000
jumlah_karyawan,10000.0,3.743000,3.308027,1.000000,2.000000,3.000000,5.000000,41.000000
has_digital_presence,10000.0,0.512100,0.499879,0.000000,0.000000,1.000000,1.000000,1.000000
omset_bulanan,10000.0,58.541112,35.038223,7.890000,34.297500,50.155000,73.205000,486.570000
populasi,10000.0,193718.647500,123225.123684,14063.000000,96887.000000,161994.000000,262565.000000,500000.000000
kepadatan_penduduk,10000.0,15784.157580,21246.981821,287.100000,2439.400000,6562.400000,20937.300000,130841.900000
income_per_kapita,10000.0,8.323852,3.511193,2.000000,5.570000,7.840000,10.540000,15.000000
jarak_ke_jalan_utama,10000.0,4.193093,4.146716,0.100000,1.130000,2.910000,5.520000,15.000000


## 2. Univariate Analysis

Exploring the distribution of individual features to understand their ranges, skewness, and potential outliers.

In [5]:
# Histograms + KDE for key numeric features
numeric_features = ['skor_potensi', 'omset_bulanan', 'income_per_kapita', 
                    'skor_infrastruktur', 'kepadatan_penduduk']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    ax = axes[i]
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue', bins=40)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    # Add mean line
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    ax.legend()

# Remove empty subplot
axes[5].set_visible(False)
plt.tight_layout()
plt.savefig('/tmp/univariate_numeric.png', bbox_inches='tight')
plt.close()
print("Numeric feature distributions plotted.")

Numeric feature distributions plotted.


In [6]:
# Bar charts for categorical features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Jenis usaha distribution
jenis_counts = df['jenis_usaha'].value_counts()
sns.barplot(x=jenis_counts.values, y=jenis_counts.index, ax=axes[0], palette='viridis')
axes[0].set_title('UMKM Count by Jenis Usaha')
axes[0].set_xlabel('Count')
for i, v in enumerate(jenis_counts.values):
    axes[0].text(v + 50, i, str(v), va='center')

# Top 10 kabupaten by UMKM count
kab_counts = df['kabupaten_kota'].value_counts().head(10)
sns.barplot(x=kab_counts.values, y=kab_counts.index, ax=axes[1], palette='magma')
axes[1].set_title('Top 10 Kabupaten/Kota by UMKM Count')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('/tmp/univariate_categorical.png', bbox_inches='tight')
plt.close()
print("Categorical feature distributions plotted.")

Categorical feature distributions plotted.


## 3. Spatial Distribution Analysis

Analyzing the geographic distribution of UMKMs across Jawa Barat, including comparisons between urban (Kota) and rural (Kabupaten) areas.

In [7]:
# Scatter plot of UMKM locations colored by skor_potensi
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
scatter = ax.scatter(df['longitude'], df['latitude'], 
                     c=df['skor_potensi'], cmap='RdYlGn', 
                     alpha=0.4, s=5)
plt.colorbar(scatter, ax=ax, label='Skor Potensi')
ax.set_title('UMKM Spatial Distribution (colored by Skor Potensi)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('/tmp/spatial_distribution.png', bbox_inches='tight')
plt.close()
print("Spatial distribution plot created.")

Spatial distribution plot created.


In [8]:
# UMKM count per kabupaten - bar chart
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
kab_all = df['kabupaten_kota'].value_counts().sort_values(ascending=True)
kab_all.plot(kind='barh', ax=ax, color='teal')
ax.set_title('UMKM Count per Kabupaten/Kota')
ax.set_xlabel('Number of UMKMs')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('/tmp/umkm_per_kabupaten.png', bbox_inches='tight')
plt.close()
print(f"Total kabupaten/kota: {df['kabupaten_kota'].nunique()}")

Total kabupaten/kota: 26


In [9]:
# Urban vs Rural comparison
# is_kota column indicates urban (True) vs rural (False)
comparison_features = ['skor_potensi', 'omset_bulanan', 'income_per_kapita', 
                       'skor_infrastruktur', 'akses_internet_pct']

fig, axes = plt.subplots(1, len(comparison_features), figsize=(20, 5))
for i, col in enumerate(comparison_features):
    sns.boxplot(data=df, x='is_kota', y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(col)
    axes[i].set_xlabel('Is Kota (Urban)')
    
plt.suptitle('Urban (Kota) vs Rural (Kabupaten) Distribution', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('/tmp/urban_vs_rural.png', bbox_inches='tight')
plt.close()

# Print summary statistics
print("=== Urban vs Rural Summary ===")
print(df.groupby('is_kota')[comparison_features].mean().round(2).T)

=== Urban vs Rural Summary ===
is_kota             False  True 
skor_potensi        50.07  75.75
omset_bulanan       54.63  71.00
income_per_kapita    7.38  11.33
skor_infrastruktur  64.66  82.32
akses_internet_pct  62.60  81.49


## 4. Correlation Analysis

Computing the full correlation matrix and identifying the strongest predictors of our target variables: `skor_potensi` and `is_survived_3yr`.

Since features were independently generated (not derived from targets), these correlations represent meaningful relationships.

In [10]:
# Full correlation matrix heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(1, 1, figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, ax=ax, 
            annot_kws={'size': 7}, square=True,
            vmin=-1, vmax=1)
ax.set_title('Correlation Matrix (Numeric Features)', fontsize=14)
plt.tight_layout()
plt.savefig('/tmp/correlation_matrix.png', bbox_inches='tight')
plt.close()
print("Correlation matrix plotted.")

Correlation matrix plotted.


In [11]:
# Top correlations with skor_potensi
print("=== Top 10 Correlations with skor_potensi ===")
skor_corr = corr_matrix['skor_potensi'].drop('skor_potensi').abs().sort_values(ascending=False)
for feat, val in skor_corr.head(10).items():
    direction = corr_matrix.loc[feat, 'skor_potensi']
    print(f"  {feat:40s} r = {direction:+.4f}")

print()
print("=== Top 10 Correlations with is_survived_3yr ===")
surv_corr = corr_matrix['is_survived_3yr'].drop('is_survived_3yr').abs().sort_values(ascending=False)
for feat, val in surv_corr.head(10).items():
    direction = corr_matrix.loc[feat, 'is_survived_3yr']
    print(f"  {feat:40s} r = {direction:+.4f}")

=== Top 10 Correlations with skor_potensi ===
  akses_internet_pct                       r = +0.8306
  skor_infrastruktur                       r = +0.8033
  income_per_kapita                        r = +0.7640
  penetrasi_kur_pct                        r = +0.5010
  risiko_gempa                             r = -0.4958
  jarak_ke_jalan_utama                     r = -0.4736
  jarak_ke_pasar                           r = -0.4150
  has_digital_presence                     r = +0.4026
  kepadatan_penduduk                       r = +0.3843
  populasi                                 r = +0.3742

=== Top 10 Correlations with is_survived_3yr ===
  has_digital_presence                     r = +0.0670
  skor_potensi                             r = +0.0566
  skor_infrastruktur                       r = +0.0394
  jarak_ke_bank_terdekat                   r = -0.0365
  akses_internet_pct                       r = +0.0355
  income_per_kapita                        r = +0.0303
  jarak_ke_jalan_utama  

## 5. Bivariate Analysis

Examining relationships between pairs of features, particularly how key variables relate to our targets.

In [12]:
# Scatter plots: skor_potensi vs key features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_pairs = [
    ('skor_infrastruktur', 'skor_potensi'),
    ('income_per_kapita', 'skor_potensi'),
    ('jumlah_kompetitor_radius_3km', 'skor_potensi')
]

for i, (x_col, y_col) in enumerate(scatter_pairs):
    ax = axes[i]
    ax.scatter(df[x_col], df[y_col], alpha=0.2, s=5, color='steelblue')
    # Add regression line
    z = np.polyfit(df[x_col], df[y_col], 1)
    p = np.poly1d(z)
    x_range = np.linspace(df[x_col].min(), df[x_col].max(), 100)
    ax.plot(x_range, p(x_range), "r--", linewidth=2, label=f'r={corr_matrix.loc[x_col, y_col]:.3f}')
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{y_col} vs {x_col}')
    ax.legend()

plt.tight_layout()
plt.savefig('/tmp/bivariate_scatter.png', bbox_inches='tight')
plt.close()
print("Bivariate scatter plots created.")

Bivariate scatter plots created.


In [13]:
# Survival rate by jenis_usaha
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

survival_by_jenis = df.groupby('jenis_usaha')['is_survived_3yr'].mean().sort_values(ascending=False)
sns.barplot(x=survival_by_jenis.values, y=survival_by_jenis.index, ax=axes[0], palette='RdYlGn')
axes[0].set_title('Survival Rate by Jenis Usaha')
axes[0].set_xlabel('Survival Rate (3yr)')
for i, v in enumerate(survival_by_jenis.values):
    axes[0].text(v + 0.005, i, f'{v:.1%}', va='center')

# Survival rate by kabupaten (top 5 and bottom 5)
survival_by_kab = df.groupby('kabupaten_kota')['is_survived_3yr'].mean().sort_values()
top5 = survival_by_kab.tail(5)
bottom5 = survival_by_kab.head(5)
combined = pd.concat([bottom5, top5])
colors = ['#d73027']*5 + ['#1a9850']*5
sns.barplot(x=combined.values, y=combined.index, ax=axes[1], palette=colors)
axes[1].set_title('Survival Rate: Top 5 & Bottom 5 Kabupaten')
axes[1].set_xlabel('Survival Rate (3yr)')
axes[1].axvline(df['is_survived_3yr'].mean(), color='black', linestyle='--', label='Overall Mean')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/survival_rates.png', bbox_inches='tight')
plt.close()
print("Survival rate analysis complete.")
print(f"Overall survival rate: {df['is_survived_3yr'].mean():.1%}")

Survival rate analysis complete.
Overall survival rate: 68.0%


In [14]:
# Box plots: omset_bulanan by is_survived_3yr
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
sns.boxplot(data=df, x='is_survived_3yr', y='omset_bulanan', ax=ax, palette='Set2')
ax.set_title('Omset Bulanan by Survival Status')
ax.set_xlabel('Survived 3 Years')
ax.set_ylabel('Omset Bulanan (Rp)')
plt.tight_layout()
plt.savefig('/tmp/omset_by_survival.png', bbox_inches='tight')
plt.close()

# Print median omset by survival
print("Median omset by survival status:")
print(df.groupby('is_survived_3yr')['omset_bulanan'].median())

Median omset by survival status:
is_survived_3yr
0    49.45
1    50.54
Name: omset_bulanan, dtype: float64


## 6. Spatial Autocorrelation

Analyzing whether UMKM potential scores show spatial clustering - i.e., are high-scoring UMKMs located near other high-scoring UMKMs?

We compute a spatial lag (average score of k-nearest neighbors) and examine the correlation with own score (conceptual Moran's I).

In [15]:
from scipy.spatial import cKDTree

# Build spatial index using coordinates
coords = df[['latitude', 'longitude']].values
tree = cKDTree(coords)

# For each UMKM, find k=10 nearest neighbors and compute average skor_potensi
k = 10
distances, indices = tree.query(coords, k=k+1)  # k+1 because first result is the point itself

# Compute spatial lag (average skor_potensi of neighbors, excluding self)
spatial_lag = np.array([df['skor_potensi'].iloc[idx[1:]].mean() for idx in indices])
df['spatial_lag_skor'] = spatial_lag

# Compute correlation (conceptual Moran's I)
morans_corr = np.corrcoef(df['skor_potensi'], spatial_lag)[0, 1]
print(f"Spatial autocorrelation (Moran's I concept): {morans_corr:.4f}")
print(f"Interpretation: {'Positive spatial clustering' if morans_corr > 0.1 else 'Weak or no spatial clustering'}")

Spatial autocorrelation (Moran's I concept): 0.9009
Interpretation: Positive spatial clustering


In [16]:
# Identify spatial hotspots and coldspots
# Hotspot: high skor_potensi AND high spatial lag (surrounded by high scores)
# Coldspot: low skor_potensi AND low spatial lag (surrounded by low scores)
median_score = df['skor_potensi'].median()
median_lag = df['spatial_lag_skor'].median()

df['spatial_cluster'] = 'Not Significant'
df.loc[(df['skor_potensi'] > median_score) & (df['spatial_lag_skor'] > median_lag), 'spatial_cluster'] = 'Hotspot (HH)'
df.loc[(df['skor_potensi'] < median_score) & (df['spatial_lag_skor'] < median_lag), 'spatial_cluster'] = 'Coldspot (LL)'
df.loc[(df['skor_potensi'] > median_score) & (df['spatial_lag_skor'] < median_lag), 'spatial_cluster'] = 'Spatial Outlier (HL)'
df.loc[(df['skor_potensi'] < median_score) & (df['spatial_lag_skor'] > median_lag), 'spatial_cluster'] = 'Spatial Outlier (LH)'

print("=== Spatial Cluster Distribution ===")
print(df['spatial_cluster'].value_counts())

# Plot Moran scatterplot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Moran scatter plot
colors_map = {'Hotspot (HH)': 'red', 'Coldspot (LL)': 'blue', 
              'Spatial Outlier (HL)': 'orange', 'Spatial Outlier (LH)': 'lightblue',
              'Not Significant': 'grey'}
for cluster, color in colors_map.items():
    mask = df['spatial_cluster'] == cluster
    axes[0].scatter(df.loc[mask, 'skor_potensi'], df.loc[mask, 'spatial_lag_skor'],
                   c=color, alpha=0.3, s=5, label=cluster)
axes[0].set_xlabel('Skor Potensi')
axes[0].set_ylabel('Spatial Lag (Avg Neighbor Score)')
axes[0].set_title(f"Moran's I Scatter Plot (r={morans_corr:.3f})")
axes[0].axhline(median_lag, color='grey', linestyle='--', alpha=0.5)
axes[0].axvline(median_score, color='grey', linestyle='--', alpha=0.5)
axes[0].legend(markerscale=3, fontsize=8)

# Spatial map of clusters
for cluster, color in colors_map.items():
    mask = df['spatial_cluster'] == cluster
    axes[1].scatter(df.loc[mask, 'longitude'], df.loc[mask, 'latitude'],
                   c=color, alpha=0.4, s=5, label=cluster)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('Spatial Clusters Map')
axes[1].legend(markerscale=3, fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/spatial_autocorrelation.png', bbox_inches='tight')
plt.close()
print("Spatial autocorrelation analysis complete.")

=== Spatial Cluster Distribution ===
spatial_cluster
Hotspot (HH)            4395
Coldspot (LL)           4391
Spatial Outlier (LH)     599
Spatial Outlier (HL)     596
Not Significant           19
Name: count, dtype: int64


Spatial autocorrelation analysis complete.


## 7. Feature Engineering

Creating new derived features that capture interactions, ratios, and composite measures that may improve model performance.

In [17]:
# Feature Engineering
print("Creating engineered features...")

# 1. Business maturity (years of operation)
df['business_maturity'] = 2024 - df['tahun_berdiri']

# 2. Infrastructure x Income interaction
df['infra_x_income'] = (df['skor_infrastruktur'] / 100) * (df['income_per_kapita'] / df['income_per_kapita'].max())

# 3. Competition density ratio
df['competition_density_ratio'] = df['jumlah_kompetitor_radius_3km'] / (df['kepadatan_penduduk'] / 1000)

# 4. Average distance to facilities
df['avg_distance_to_facilities'] = (df['jarak_ke_jalan_utama'] + df['jarak_ke_pasar'] + df['jarak_ke_bank_terdekat']) / 3

# 5. Market gap score (market opportunity)
df['market_gap_score'] = df['populasi'] / (df['jumlah_kompetitor_radius_3km'] + 1)

# 6. Digital readiness index
df['digital_readiness_index'] = (df['akses_internet_pct'] / 100 + df['has_digital_presence']) / 2

# 7. Risk composite
df['risk_composite'] = 0.5 * df['risiko_banjir'] + 0.5 * df['risiko_gempa']

# 8. Financial access score
df['financial_access_score'] = (1 - df['jarak_ke_bank_terdekat'] / df['jarak_ke_bank_terdekat'].max()) * 0.5 + (df['penetrasi_kur_pct'] / 100) * 0.5

# 9. Omset per karyawan (productivity)
df['omset_per_karyawan'] = df['omset_bulanan'] / df['jumlah_karyawan']

# 10. Location advantage
df['location_advantage'] = df['skor_infrastruktur'] * (1 - df['risk_composite'])

print(f"Total features after engineering: {len(df.columns)}")
print()
print("=== New Features Summary ===")
new_features = ['business_maturity', 'infra_x_income', 'competition_density_ratio',
                'avg_distance_to_facilities', 'market_gap_score', 'digital_readiness_index',
                'risk_composite', 'financial_access_score', 'omset_per_karyawan', 'location_advantage']
print(df[new_features].describe().T)

Creating engineered features...
Total features after engineering: 36

=== New Features Summary ===
                              count          mean           std          min  \
business_maturity           10000.0      6.971800      4.329004     0.000000   
infra_x_income              10000.0      0.397881      0.218480     0.044400   
competition_density_ratio   10000.0      0.691259      1.585044     0.000000   
avg_distance_to_facilities  10000.0      4.992061      3.323915     0.453333   
market_gap_score            10000.0  70469.881677  78342.629249  2401.166667   
digital_readiness_index     10000.0      0.591623      0.277165     0.150000   
risk_composite              10000.0      0.296797      0.087357     0.141000   
financial_access_score      10000.0      0.531714      0.127557     0.026000   
omset_per_karyawan          10000.0     27.290672     27.447130     0.708667   
location_advantage          10000.0     48.923238     14.342786    13.603000   

                    

## 8. Feature Selection

Using mutual information scores and Variance Inflation Factor (VIF) analysis to select the most informative, non-redundant features for modeling.

In [18]:
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Prepare feature matrix (numeric only, exclude targets and spatial_cluster/spatial_lag)
exclude_cols = ['skor_potensi', 'is_survived_3yr', 'spatial_lag_skor', 'spatial_cluster']
feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns if col not in exclude_cols]

X = df[feature_cols].copy()
# Handle any infinite values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Mutual Information for skor_potensi (continuous target)
mi_potensi = mutual_info_regression(X, df['skor_potensi'], random_state=42)
mi_potensi_df = pd.DataFrame({'Feature': feature_cols, 'MI_Score': mi_potensi}).sort_values('MI_Score', ascending=False)

print("=== Mutual Information Scores vs skor_potensi (Top 15) ===")
print(mi_potensi_df.head(15).to_string(index=False))

=== Mutual Information Scores vs skor_potensi (Top 15) ===
                   Feature  MI_Score
   digital_readiness_index  5.767217
        location_advantage  5.653406
        kepadatan_penduduk  5.650089
            infra_x_income  5.647101
          market_gap_score  5.620144
                  populasi  5.534085
    financial_access_score  5.515644
avg_distance_to_facilities  5.512637
    jarak_ke_bank_terdekat  5.264251
            risk_composite  5.217906
        akses_internet_pct  5.175129
        skor_infrastruktur  5.166916
              risiko_gempa  5.137937
         income_per_kapita  5.122566
             risiko_banjir  5.089439


In [19]:
# Mutual Information for is_survived_3yr (binary target)
mi_survival = mutual_info_classif(X, df['is_survived_3yr'], random_state=42)
mi_survival_df = pd.DataFrame({'Feature': feature_cols, 'MI_Score': mi_survival}).sort_values('MI_Score', ascending=False)

print("=== Mutual Information Scores vs is_survived_3yr (Top 15) ===")
print(mi_survival_df.head(15).to_string(index=False))

=== Mutual Information Scores vs is_survived_3yr (Top 15) ===
                     Feature  MI_Score
            market_gap_score  0.008346
               omset_bulanan  0.007247
   competition_density_ratio  0.007145
               risiko_banjir  0.006140
                risiko_gempa  0.005224
jumlah_kompetitor_radius_3km  0.002721
             jumlah_karyawan  0.002376
                    latitude  0.002254
              infra_x_income  0.001505
          akses_internet_pct  0.001394
              risk_composite  0.000913
        has_digital_presence  0.000775
  avg_distance_to_facilities  0.000562
              jarak_ke_pasar  0.000000
           income_per_kapita  0.000000


In [20]:
# VIF Analysis for top features
# Select top features by mutual information for skor_potensi
top_features = mi_potensi_df.head(15)['Feature'].tolist()

X_vif = df[top_features].copy()
X_vif = X_vif.replace([np.inf, -np.inf], np.nan)
X_vif = X_vif.fillna(X_vif.median())

# Compute VIF
vif_data = pd.DataFrame()
vif_data['Feature'] = top_features
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(len(top_features))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("=== Variance Inflation Factor (VIF) Analysis ===")
print("(VIF > 10 indicates high multicollinearity)")
print()
print(vif_data.to_string(index=False))

# Identify high-VIF features to potentially drop
high_vif = vif_data[vif_data['VIF'] > 10]['Feature'].tolist()
if high_vif:
    print(f"\nHigh VIF features (consider dropping): {high_vif}")
else:
    print("\nNo features with VIF > 10. All features are acceptable.")

=== Variance Inflation Factor (VIF) Analysis ===
(VIF > 10 indicates high multicollinearity)

                   Feature         VIF
             risiko_banjir         inf
              risiko_gempa         inf
            risk_composite         inf
        skor_infrastruktur 2191.034960
        location_advantage 1196.681750
    financial_access_score  171.569944
            infra_x_income  153.568643
         income_per_kapita  150.628128
        akses_internet_pct  140.141971
avg_distance_to_facilities   67.821236
    jarak_ke_bank_terdekat   39.495974
                  populasi    8.319349
   digital_readiness_index    7.326615
        kepadatan_penduduk    4.111038
          market_gap_score    2.486541

High VIF features (consider dropping): ['risiko_banjir', 'risiko_gempa', 'risk_composite', 'skor_infrastruktur', 'location_advantage', 'financial_access_score', 'infra_x_income', 'income_per_kapita', 'akses_internet_pct', 'avg_distance_to_facilities', 'jarak_ke_bank_terdekat']


In [21]:
# Final recommended feature set
# Keep features with VIF <= 10, or if all have high VIF, keep the most informative ones
recommended_features = vif_data[vif_data['VIF'] <= 10]['Feature'].tolist()

# If too many were dropped, add back the most important ones
if len(recommended_features) < 8:
    # Add features sorted by MI score that aren't already included
    for feat in mi_potensi_df['Feature'].tolist():
        if feat not in recommended_features and len(recommended_features) < 12:
            recommended_features.append(feat)

print("=== Final Recommended Feature Set for Modeling ===")
print(f"Number of features: {len(recommended_features)}")
print()
for i, feat in enumerate(recommended_features, 1):
    mi_score = mi_potensi_df[mi_potensi_df['Feature'] == feat]['MI_Score'].values
    vif_score = vif_data[vif_data['Feature'] == feat]['VIF'].values
    mi_str = f"MI={mi_score[0]:.4f}" if len(mi_score) > 0 else "MI=N/A"
    vif_str = f"VIF={vif_score[0]:.2f}" if len(vif_score) > 0 else "VIF=N/A"
    print(f"  {i:2d}. {feat:35s} {mi_str}  {vif_str}")

=== Final Recommended Feature Set for Modeling ===
Number of features: 12

   1. populasi                            MI=5.5341  VIF=8.32
   2. digital_readiness_index             MI=5.7672  VIF=7.33
   3. kepadatan_penduduk                  MI=5.6501  VIF=4.11
   4. market_gap_score                    MI=5.6201  VIF=2.49
   5. location_advantage                  MI=5.6534  VIF=1196.68
   6. infra_x_income                      MI=5.6471  VIF=153.57
   7. financial_access_score              MI=5.5156  VIF=171.57
   8. avg_distance_to_facilities          MI=5.5126  VIF=67.82
   9. jarak_ke_bank_terdekat              MI=5.2643  VIF=39.50
  10. risk_composite                      MI=5.2179  VIF=inf
  11. akses_internet_pct                  MI=5.1751  VIF=140.14
  12. skor_infrastruktur                  MI=5.1669  VIF=2191.03


## 9. Save Engineered Dataset

Saving the complete dataset with all original and engineered features for use in subsequent modeling notebooks.

In [22]:
# Drop temporary columns used for spatial analysis
df_save = df.drop(columns=['spatial_lag_skor', 'spatial_cluster'], errors='ignore')

# Save to CSV
output_path = '../data/umkm_engineered.csv'
df_save.to_csv(output_path, index=False)

print(f"Engineered dataset saved to: {output_path}")
print(f"Shape: {df_save.shape}")
print(f"Total columns: {len(df_save.columns)}")
print()
print("=== Column List ===")
for i, col in enumerate(df_save.columns, 1):
    print(f"  {i:2d}. {col}")

Engineered dataset saved to: ../data/umkm_engineered.csv
Shape: (10000, 34)
Total columns: 34

=== Column List ===
   1. kabupaten_kota
   2. kecamatan
   3. latitude
   4. longitude
   5. is_kota
   6. jenis_usaha
   7. tahun_berdiri
   8. jumlah_karyawan
   9. has_digital_presence
  10. omset_bulanan
  11. populasi
  12. kepadatan_penduduk
  13. income_per_kapita
  14. jarak_ke_jalan_utama
  15. jarak_ke_pasar
  16. akses_internet_pct
  17. skor_infrastruktur
  18. jumlah_kompetitor_radius_3km
  19. jarak_ke_bank_terdekat
  20. penetrasi_kur_pct
  21. risiko_banjir
  22. risiko_gempa
  23. skor_potensi
  24. is_survived_3yr
  25. business_maturity
  26. infra_x_income
  27. competition_density_ratio
  28. avg_distance_to_facilities
  29. market_gap_score
  30. digital_readiness_index
  31. risk_composite
  32. financial_access_score
  33. omset_per_karyawan
  34. location_advantage


In [23]:
# Summary of new engineered features
print("=== Engineered Features Summary ===")
new_features = ['business_maturity', 'infra_x_income', 'competition_density_ratio',
                'avg_distance_to_facilities', 'market_gap_score', 'digital_readiness_index',
                'risk_composite', 'financial_access_score', 'omset_per_karyawan', 'location_advantage']

for feat in new_features:
    print(f"\n{feat}:")
    print(f"  Mean: {df_save[feat].mean():.4f}, Std: {df_save[feat].std():.4f}")
    print(f"  Min: {df_save[feat].min():.4f}, Max: {df_save[feat].max():.4f}")
    print(f"  Median: {df_save[feat].median():.4f}")

print(f"\n{'='*50}")
print(f"Notebook 02 complete. Dataset ready for modeling.")
print(f"Original features: 24, New features: {len(new_features)}, Total: {len(df_save.columns)}")

=== Engineered Features Summary ===

business_maturity:
  Mean: 6.9718, Std: 4.3290
  Min: 0.0000, Max: 14.0000


  Median: 7.0000

infra_x_income:
  Mean: 0.3979, Std: 0.2185
  Min: 0.0444, Max: 0.9500
  Median: 0.3501

competition_density_ratio:
  Mean: 0.6913, Std: 1.5850
  Min: 0.0000, Max: 28.5863
  Median: 0.2799

avg_distance_to_facilities:
  Mean: 4.9921, Std: 3.3239
  Min: 0.4533, Max: 18.4500
  Median: 4.2233

market_gap_score:
  Mean: 70469.8817, Std: 78342.6292
  Min: 2401.1667, Max: 500000.0000
  Median: 38630.7500

digital_readiness_index:
  Mean: 0.5916, Std: 0.2772
  Min: 0.1500, Max: 0.9750
  Median: 0.7117

risk_composite:
  Mean: 0.2968, Std: 0.0874
  Min: 0.1410, Max: 0.5145
  Median: 0.2925

financial_access_score:
  Mean: 0.5317, Std: 0.1276
  Min: 0.0260, Max: 0.7184
  Median: 0.5700

omset_per_karyawan:
  Mean: 27.2907, Std: 27.4471
  Min: 0.7087, Max: 307.7600
  Median: 18.0300

location_advantage:
  Mean: 48.9232, Std: 14.3428
  Min: 13.6030, Max: 77.0792
  Median: 48.4962

Notebook 02 complete. Dataset ready for modeling.
Original features: 24, New features: 10, Total: 3